# Synthetic 50 MP RAW - PRD v6 (single run, 95% gate)

One **Runtime > Run all** makes three synthetic Sony A1 II style RAW outputs from rendered images and scores each one
against the real RAW they came from:

| Output | Size | DNG | 16-bit TIFF for Photoshop |
|---|---|---|---|
| 2K | 1920 x 1280 | LinearRaw | developed, sRGB ICC |
| 4K | 3840 x 2560 | LinearRaw | developed, sRGB ICC |
| 50 MP | 8640 x 5760 | RGGB Bayer mosaic with A1 II shot and read noise | developed, sRGB ICC |

Each output passes only when every colour plane reaches **A >= 95%** of the reference RAW (and P2 >= 95%). The verdict
is PASS (all three pass), MISSED (it says which output, which plane and what cost the most) or INVALID (a check or step
failed; it says why). The reference is one real A1 II frame, so the score is in-sample (PRD v6 section 13).

## Before you press Run all

1. **Runtime > Change runtime type: A100 GPU with High-RAM** (Colab Pro or Pro+). L4 runs with a warning and is slower;
   T4 and CPU-only runtimes are rejected at preflight before any heavy work.
2. Expect about **2.5 to 4.5 hours** on an A100. Keep the tab open.
3. Leave the settings below as they are for the maximum-quality benchmark run. Nothing asks for input.

## What arrives

When the run ends (also after a failure) the last cell zips the results and downloads them: first
`<run_id>_01_summary.zip` (report.html, verdict, metrics, error maps), then one archive per output. Chrome may ask to
allow multiple downloads for colab.research.google.com: allow it. If a download was blocked, the tab was closed or the
kernel crashed, the driver cell prints a one-line recovery command at the start; paste it into a new cell to rebuild and
download every archive. Files on the runtime are lost when Colab recycles it unless `DRIVE_MIRROR = True`.

In [ ]:
# Settings. The defaults are the maximum-quality run; Run all needs no edits (PRD v6 section 9.2).
PRESET = "max"  # the only preset: every stage that helps accuracy on, at its highest setting (section 9.3)
REPO_REF = "19fe2f0a3abfd5570d560f7f81899156fb9d1edd"  # pinned commit of the code this notebook runs
SOURCE_4K_URL = None  # optional direct URL of your own 4K image: adds its 2K, 4K and 50 MP outputs (no accuracy claim)
SOURCE_2K_URL = None  # optional direct URL of your own 2K image, likewise
REFERENCE_ARW_URL = None  # None uses the pinned publisher URL; another URL must serve the same SHA-256
DOWNLOAD_ALL = False  # True adds one more archive with the whole run folder (several GB)
DRIVE_MIRROR = False  # True mounts Google Drive first (one prompt) and mirrors the summary after every step

## Driver

Runs steps 1-6 (Drive mount if enabled, setup, preflight, benchmark, production, sweeps) inside one `try/finally`,
so packaging and downloads happen after success and after any failure. No edits needed.

In [ ]:
# Driver: steps 1-6 inside one try/finally; the finally block packages and downloads (PRD v6 section 9.2).
import datetime
import hashlib
import json
import subprocess
import sys
import traceback
import zipfile
from pathlib import Path

try:
    import google.colab  # noqa: F401
except ImportError as error:
    raise RuntimeError("This notebook runs only in Google Colab (PRD v6 section 9.1)") from error

SETTINGS = {
    "PRESET": PRESET,
    "REPO_REF": REPO_REF,
    "SOURCE_4K_URL": SOURCE_4K_URL,
    "SOURCE_2K_URL": SOURCE_2K_URL,
    "REFERENCE_ARW_URL": REFERENCE_ARW_URL,
    "DOWNLOAD_ALL": DOWNLOAD_ALL,
    "DRIVE_MIRROR": DRIVE_MIRROR,
}
REPO_URL = "https://github.com/RDC28/experiment_50MP.git"
CLONE = Path("/content/experiment_50MP")
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + REPO_REF[:7]
RUN_DIR = Path("/content/exp50mp_v6") / RUN_ID
RUN_DIR.mkdir(parents=True)
CONFIG = {"run_id": RUN_ID, "clone": str(CLONE), "settings": SETTINGS}
(RUN_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2))
print(f"run_dir: {RUN_DIR}")
print(f"clone: {CLONE}")
print("If the downloads are blocked or the kernel crashes, run this in a new cell:")
print(
    f'import sys; sys.path.insert(0, "{CLONE}"); from prd_v6.delivery import download_results; '
    f'download_results("{RUN_DIR}")',
    flush=True,
)


def _now():
    return datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def _mark(step, name, *, started=False, error=None):
    """Steps 1 and 2 run before the package exists; same state.json format as prd_v6.delivery."""
    path = RUN_DIR / "state.json"
    state = json.loads(path.read_text()) if path.exists() else {"steps": []}
    record = next((r for r in state["steps"] if r["step"] == step), None)
    if record is None:
        record = {"step": step, "name": name}
        state["steps"].append(record)
    if started:
        state["step_started"] = step
        record.update(status="started", started_at=_now())
    else:
        record.update(status="failed" if error else "finished", finished_at=_now())
        if error is not None:
            record["error"] = f"{type(error).__name__}: {error}"
            record["traceback"] = "".join(traceback.format_exception(type(error), error, error.__traceback__))
    path.write_text(json.dumps(state, indent=2))
    line = f"{_now()} step {step} {record['status']}: {name}"
    print(line, flush=True)
    with open(RUN_DIR / "run.log", "a") as handle:
        handle.write(line + "\n")


def _setup():
    git = ["git", "-C", str(CLONE)]
    if not (CLONE / ".git").exists():
        subprocess.run(["git", "clone", "--filter=blob:none", REPO_URL, str(CLONE)], check=True)
    subprocess.run(git + ["fetch", "origin", REPO_REF], check=True)
    subprocess.run(git + ["checkout", "--detach", REPO_REF], check=True)
    head = subprocess.check_output(git + ["rev-parse", "HEAD"], text=True).strip()
    if head != REPO_REF:
        raise RuntimeError(f"checked out {head}, expected REPO_REF {REPO_REF}")
    for name in [m for m in sys.modules if m.split(".")[0] in {"prd_v6", "prd_v5", "synthraw", "synthetic_raw_lab"}]:
        del sys.modules[name]
    sys.path.insert(0, str(CLONE))
    from prd_v6 import install

    install.run(RUN_DIR, CLONE)


def _fallback(reason):
    """Minimal packaging when the clone or its delivery module is unavailable (standard library only)."""
    archive = RUN_DIR.parent / f"{RUN_ID}_fallback.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
        for path in sorted(RUN_DIR.rglob("*")):
            if path.is_file() and path.suffix.lower() not in {".arw", ".pth", ".part"}:
                bundle.write(path, path.relative_to(RUN_DIR.parent))
    digest = hashlib.sha256(archive.read_bytes()).hexdigest()
    print(f"download {archive.name}  {archive.stat().st_size / 1e6:.1f} MB  sha256 {digest}", flush=True)
    try:
        from google.colab import files

        files.download(str(archive))
    except Exception as error:  # noqa: BLE001
        print(f"download failed ({type(error).__name__}: {error}); use the Files pane: {archive}")
    rule = "=" * 78
    print(f"{rule}\nPRD v6 RUN {RUN_ID}: INVALID\nINVALID: {reason}\nRun folder: {RUN_DIR}\n{rule}", flush=True)


FAILURE = None
try:
    if DRIVE_MIRROR:
        _mark(1, "Drive mount", started=True)
        try:
            from google.colab import drive

            drive.mount("/content/drive")
        except BaseException as error:
            _mark(1, "Drive mount", error=error)
            raise
        _mark(1, "Drive mount")
    _mark(2, "Setup", started=True)
    try:
        _setup()
    except BaseException as error:
        _mark(2, "Setup", error=error)
        raise
    _mark(2, "Setup")
    from prd_v6 import delivery, driver

    delivery.checkpoint(RUN_DIR, SETTINGS)
    driver.run(RUN_DIR, SETTINGS)
except BaseException as error:
    state = json.loads((RUN_DIR / "state.json").read_text()) if (RUN_DIR / "state.json").exists() else {}
    FAILURE = {"step": state.get("step_started"), "error": f"{type(error).__name__}: {error}"}
    traceback.print_exc()
finally:
    try:
        from prd_v6.delivery import finalize_run

        VERDICT, ARCHIVES = finalize_run(RUN_DIR, SETTINGS, FAILURE)
    except Exception as error:  # noqa: BLE001 - the fallback must still deliver something
        traceback.print_exc()
        _fallback(f"{FAILURE['error'] if FAILURE else 'packaging failed'}; packaging: {type(error).__name__}: {error}")